In [ ]:
import os
from dotenv import load_dotenv
from datasets import load_dataset

"""
#made dataset public no need of huggingface_key
load_dotenv()
huggingface_key = os.getenv("HUGGINGFACE_KEY")

if not huggingface_key:
    raise ValueError("HUGGINGFACE_KEY not found — add it to your .env file")
"""

REPO_ID = "sujalgawas/jev-1min-stock-bars"

dataset = load_dataset(REPO_ID)

df = dataset["train"].to_pandas()

In [ ]:
df.head()

In [ ]:
import pandas as pd
from torch.utils.data import Dataset
import torch
import numpy as np

class JEPATimeSeriesDataset(Dataset):
    """
    Slices past context window (X) and future target window (Y) on-the-fly.
    Prevents symbol crossing and memory duplication.
    """
    def __init__(
        self,
        df: pd.DataFrame,
        past_window: int = 60,
        future_horizon: int = 15,
        feature_cols: list = ["log_return", "bar_range", "vwap_dist", "log_vol_change"]
    ):
        self.past_window = past_window
        self.future_horizon = future_horizon
        self.feature_cols = feature_cols

        self.valid_indices = []
        self.data_by_symbol = {}

        # Separate data by symbol to build valid contiguous sequence boundaries
        for symbol, group in df.groupby("symbol"):
            features = group[feature_cols].values.astype(np.float32)
            prices = group["close"].values.astype(np.float32)

            n_samples = len(features) - (past_window + future_horizon) + 1
            if n_samples > 0:
                self.data_by_symbol[symbol] = {
                    "features": features,
                    "prices": prices
                }
                for i in range(n_samples):
                    self.valid_indices.append((symbol, i))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        symbol, start_i = self.valid_indices[idx]
        features = self.data_by_symbol[symbol]["features"]

        # X_past: Window from t-60 to t  -> Shape: [past_window, n_features]
        x_past = features[start_i : start_i + self.past_window]

        # Y_future: Horizon from t to t+15 -> Shape: [future_horizon, n_features]
        y_future = features[start_i + self.past_window : start_i + self.past_window + self.future_horizon]

        return torch.tensor(x_past), torch.tensor(y_future)

In [ ]:
import torch.nn as nn
import torch

class PatchEmbed(nn.Module):
  def __init__(self,n_features:int,patch_len:int, dim:int):
    super().__init__()
    self.patch_len = patch_len
    self.n_features = n_features
    self.proj = nn.Linear(patch_len * n_features, dim)

  def forward(self, x : torch.Tensor) -> torch.Tensor:
    # (B,L,F) -> (B,N,F * P) -> (B,N,D)  where N = L // P
    B,L,F = x.shape
    P = self.patch_len
    assert L % P == 0, f"window{L} not divisible by patch_len {P}"
    N = L // P
    x = x.reshape(B,N, P * F)
    return self.proj(x)

In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self, dim:int, num_layers:int,heads: int, mlp_ratio: float =4.0, dropout: float = 0.0):
    super().__init__()

    layer = nn.TransformerEncoderLayer(
        d_model = dim, nhead= heads, dim_feedforward= int(dim * mlp_ratio),
        dropout=dropout, activation="gelu", batch_first = True, norm_first=True,
    )
    self.blocks = nn.TransformerEncoder(layer,num_layers=num_layers, enable_nested_tensor=False)
    self.norm = nn.LayerNorm(dim)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.norm(self.blocks(x))

In [ ]:
class LOBEncoder(nn.Module):
  def __init__(self, n_features:int, window: int, patch_len: int = 8,
               dim : int = 192, num_layers: int = 6, heads: int = 6,
               mlp_ratio: float = 4.0, dropout: float = 0.0):
    super().__init__()
    self.n_patches = window // patch_len
    self.dim = dim
    self.patch_embed = PatchEmbed(n_features, patch_len, dim)
    self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches,dim))
    nn.init.trunc_normal_(self.pos_embed, std=0.02)
    self.encoder = TransformerEncoder(dim, num_layers, heads, mlp_ratio, dropout)

  def embed_tokens(self, x: torch.Tensor) -> torch.Tensor:
    #pre-encoder
    return self.patch_embed(x) + self.pos_embed

  def forward(self, x:torch.Tensor, idx:torch.Tensor | None = None) -> torch.Tensor:
    tokens = self.embed_tokens(x)
    if idx is not None:
      # gather visible positions (shared mask)
      tokens = tokens[:, idx, :]
    return self.encoder(tokens)

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class JEPAPredictor(nn.Module):
  def __init__(self, dim:int, n_patches:int, pred_dim : int=128,
               num_layer : int = 4, heads:int = 4, mlp_ratio: float = 4.0):
    super().__init__()
    self.input_proj = nn.Linear(dim, pred_dim)
    self.output_proj = nn.Linear(pred_dim, dim)
    self.mask_token = nn.Parameter(torch.zeros(1,1,pred_dim))
    self.pos_embed = nn.Parameter(torch.zeros(1, n_patches,pred_dim))
    nn.init.trunc_normal_(self.pos_embed,std=0.02)
    nn.init.trunc_normal_(self.mask_token,std=0.02)
    self.encoder = TransformerEncoder(pred_dim,num_layer, heads,mlp_ratio)

  def forward(self, ctx_reps: torch.Tensor, ctx_idx: torch.Tensor,
              tgt_idx: torch.Tensor) -> torch.Tensor:
    proj = self.input_proj(ctx_reps)
    B = proj.shape[0]
    N = self.pos_embed.shape[1]
    seq = self.mask_token.to(proj.dtype).expand(B,N,-1).clone()
    seq[:, ctx_idx, :] = proj
    seq = self.encoder(seq)
    return self.output_proj(seq[:,tgt_idx,:])

class JEPA(nn.Module):
  def __init__(self, n_features: int, window: int, patch_len: int = 8,
               dim: int = 192, num_layer: int=6, heads: int = 6, pred_dim: int = 128,
               mlp_ratio: float = 4.0, dropout: float = 0.0):
    super().__init__()
    self.context_encoder = LOBEncoder(n_features, window, patch_len, pred_dim, num_layer,
                                      heads, mlp_ratio, dropout)
    self.target_encoder = copy.deepcopy(self.context_encoder)

    for p in self.target_encoder.parameters():
      p.requires_grad_(False)
    self.n_patches = self.context_encoder.n_patches
    self.predictor = JEPAPredictor(dim, self.n_patches,dim,num_layer,
                                   heads, mlp_ratio)
    self.dim = dim

  @torch.no_grad()
  def update_target(self, momentum: float) -> None:
    for pe, pt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
      pt.mul_(momentum).add_(pe.detach(), alpha=1.0 - momentum)

  def forward(self, x: torch.Tensor, ctx_idx: torch.Tensor, tgt_idx: torch.Tensor):
    with torch.no_grad():
      t_all = self.target_encoder(x, idx=None)         # (B, N, D)
      tgt = t_all[:, tgt_idx, :]
      tgt = F.layer_norm(tgt, (tgt.shape[-1],))        # I-JEPA target norm
    ctx = self.context_encoder(x, idx=ctx_idx)           # (B, Nc, D)
    pred = self.predictor(ctx, ctx_idx, tgt_idx)         # (B, Nt, D)
    loss = F.smooth_l1_loss(pred, tgt)
    with torch.no_grad():
      flat_t = tgt.reshape(-1, tgt.shape[-1])
      flat_p = pred.reshape(-1, pred.shape[-1])
      metrics = {
          "loss": loss.detach(),
          "tgt_std": flat_t.std(dim=0).mean(),         # collapse -> 0
          "pred_std": flat_p.std(dim=0).mean(),
      }
    return loss, metrics

  @torch.no_grad()
  def energy(self, x: torch.Tensor, ctx_idx: torch.Tensor, tgt_idx: torch.Tensor) -> torch.Tensor:
    t_all = self.target_encoder(x, idx=None)
    tgt = F.layer_norm(t_all[:, tgt_idx, :], (t_all.shape[-1],))
    ctx = self.context_encoder(x, idx=ctx_idx)
    pred = self.predictor(ctx, ctx_idx, tgt_idx)
    return F.smooth_l1_loss(pred, tgt, reduction="none").mean(dim=(1, 2))

  @torch.no_grad()
  def represent(self, x: torch.Tensor, pool: str = "mean") -> torch.Tensor:
    reps = self.context_encoder(x, idx=None)             # (B, N, D)
    if pool == "mean":
      return reps.mean(dim=1)
    if pool == "last":
      return reps[:, -1, :]
    return reps.reshape(reps.shape[0], -1)               # 'concat'


def sample_mask(variant: str, n_patches: int, ctx_frac: float = 0.5,
                min_block: int = 2, max_block: int = 6,
                rng: np.random.Generator | None = None,
                min_ctx: int | None = None, max_ctx: int | None = None) -> tuple[np.ndarray, np.ndarray]:
  rng = rng or np.random.default_rng()
  if variant == "causal":
    if min_ctx is not None:
      hi = min(n_patches - 1, max_ctx if max_ctx is not None else n_patches - 1)
      nc = int(rng.integers(max(1, min_ctx), hi + 1))
    else:
      nc = max(1, min(n_patches - 1, int(round(n_patches * ctx_frac))))
    return np.arange(nc), np.arange(nc, n_patches)
  if variant == "block":
    blk = int(rng.integers(min_block, max_block + 1))
    blk = min(blk, n_patches - 1)
    start = int(rng.integers(0, n_patches - blk + 1))
    tgt = np.arange(start, start + blk)
    ctx = np.setdiff1d(np.arange(n_patches), tgt)
    return ctx, tgt
  raise ValueError(f"unknown mask variant {variant}")


def momentum_schedule(step: int, total_steps: int, base: float = 0.996,
                      final: float = 1.0) -> float:
  if total_steps <= 0:
    return base
  frac = min(1.0, step / total_steps)
  return final - (final - base) * 0.5 * (1 + np.cos(np.pi * frac))

In [ ]:
from torch.utils.data import DataLoader, random_split

dataset = JEPATimeSeriesDataset(
    df=df,
    past_window=60,
    future_horizon=15
)

length = len(dataset)

train_size = int(length * 0.8)
val_size = length - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
import os
import math
import time
import json
from contextlib import nullcontext

import numpy as np
import torch
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"device: cuda  ({torch.cuda.get_device_name(0)})")
else:
    print("device: cpu — no GPU detected by torch.cuda.is_available().")
    print("This is a Colab runtime setting, not something the code controls:")
    print("  Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4) -> Save")
    print("then Runtime -> Restart session and re-run cells from the top.")
    print("If you already selected a GPU runtime and still see this, torch may have")
    print("installed CPU-only — check with: !python -c \"import torch; print(torch.__version__)\"")

In [ ]:
# ---- config ----
FEATURE_COLS = ["log_return", "bar_range", "vwap_dist", "log_vol_change"]
PAST_WINDOW = 60
FUTURE_HORIZON = 15
PATCH_LEN = 5          # must evenly divide BOTH past_window and future_horizon so the
                       # context/target patch boundary lines up exactly with the real
                       # past/future split (patch_len=8 does NOT divide 60, 15 or 75 cleanly)
N_FEATURES = len(FEATURE_COLS)
WINDOW = PAST_WINDOW + FUTURE_HORIZON   # 75 timesteps -> 15 patches of length 5

EPOCHS = 1
LR = 3e-4
WEIGHT_DECAY = 0.05
GRAD_CLIP = 1.0
WARMUP_FRAC = 0.05          # fraction of total steps spent on LR warmup
EMA_BASE = 0.996
EMA_FINAL = 1.0
VAL_EVERY = 1               # validate every N epochs
OUT_DIR = "checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
MODEL_CONFIG = dict(
    n_features=N_FEATURES,
    window=WINDOW,
    patch_len=PATCH_LEN,
    dim=192,
    num_layer=6,
    heads=6,
    pred_dim=192,
    mlp_ratio=4.0,
    dropout=0.0,
)

model = JEPA(**MODEL_CONFIG).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"JEPA params: {n_params/1e6:.2f}M   n_patches: {model.n_patches}")
assert model.n_patches == WINDOW // PATCH_LEN

In [ ]:
# Fixed causal split: the dataset already separates past (context) from future
# (target) for us, so we don't need sample_mask's random causal split here —
# context is always the first PAST_WINDOW//PATCH_LEN patches, target is the rest.
assert PAST_WINDOW % PATCH_LEN == 0 and FUTURE_HORIZON % PATCH_LEN == 0, \
    "PATCH_LEN must evenly divide both past_window and future_horizon"

N_CTX_PATCHES = PAST_WINDOW // PATCH_LEN
ctx_idx = torch.arange(N_CTX_PATCHES, device=device)
tgt_idx = torch.arange(N_CTX_PATCHES, model.n_patches, device=device)
print("context patches:", ctx_idx.numel(), " target patches:", tgt_idx.numel())

In [ ]:
def param_groups(model, weight_decay):
    """No weight decay on biases, norm params, pos_embed or mask_token."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim < 2 or "pos_embed" in name or "mask_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def lr_factor(step, warmup_steps, total_steps):
    """Linear warmup then cosine decay to 0."""
    if step < warmup_steps:
        return (step + 1) / max(1, warmup_steps)
    frac = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, frac)))

In [ ]:
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))

optimizer = torch.optim.AdamW(
    param_groups(model, WEIGHT_DECAY), lr=LR, betas=(0.9, 0.95)
)

amp_ctx = (
    torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)

print(f"steps/epoch: {steps_per_epoch}  total steps: {total_steps}  warmup: {warmup_steps}")

In [ ]:
@torch.no_grad()
def validate():
    model.eval()
    losses = []
    for x_past, y_future in val_loader:
        x = torch.cat([x_past, y_future], dim=1).to(device, non_blocking=True)
        with amp_ctx:
            loss, _ = model(x, ctx_idx, tgt_idx)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses)) if losses else float("nan")

## Run training

In [ ]:
history = []
best_val = float("inf")
step = 0
model.train()
t0 = time.time()

for epoch in range(EPOCHS):
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    for x_past, y_future in pbar:
        # dataset gives us past/future separately; JEPA.forward expects one
        # window and splits it internally via ctx_idx/tgt_idx
        x = torch.cat([x_past, y_future], dim=1).to(device, non_blocking=True)

        lr = LR * lr_factor(step, warmup_steps, total_steps)
        for g in optimizer.param_groups:
            g["lr"] = lr

        with amp_ctx:
            loss, metrics = model(x, ctx_idx, tgt_idx)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        model.update_target(momentum_schedule(step, total_steps, EMA_BASE, EMA_FINAL))

        step += 1
        pbar.set_postfix(loss=metrics["loss"].item(), tgt_std=metrics["tgt_std"].item(),
                          pred_std=metrics["pred_std"].item(), lr=lr)

    history.append(dict(epoch=epoch, step=step, lr=lr,
                        loss=metrics["loss"].item(),
                        tgt_std=metrics["tgt_std"].item(),
                        pred_std=metrics["pred_std"].item()))

    if (epoch + 1) % VAL_EVERY == 0 or epoch == EPOCHS - 1:
        vloss = validate()
        history[-1]["val_loss"] = vloss
        wps = step * train_loader.batch_size / (time.time() - t0)
        print(f"epoch {epoch+1:3d}/{EPOCHS}  train_loss {metrics['loss'].item():.4f}  "
              f"val_loss {vloss:.4f}  (best {best_val:.4f})  {wps:.0f} win/s")

        ckpt = dict(model=model.state_dict(), epoch=epoch, step=step, val_loss=vloss,
                    config=MODEL_CONFIG)
        torch.save(ckpt, os.path.join(OUT_DIR, "last.pt"))
        if vloss < best_val:
            best_val = vloss
            torch.save(ckpt, os.path.join(OUT_DIR, "best.pt"))

json.dump(history, open(os.path.join(OUT_DIR, "history.json"), "w"), indent=2)
print(f"done. best val loss: {best_val:.4f}")

epoch 1/1:   0%|          | 0/216521 [00:01<?, ?it/s]

## Loss curve

In [ ]:
import matplotlib.pyplot as plt

epochs_x = [h["epoch"] for h in history]
train_losses = [h["loss"] for h in history]
val_losses = [h.get("val_loss") for h in history]

plt.figure(figsize=(8, 4))
plt.plot(epochs_x, train_losses, label="train loss")
val_x = [e for e, v in zip(epochs_x, val_losses) if v is not None]
val_y = [v for v in val_losses if v is not None]
plt.plot(val_x, val_y, label="val loss", marker="o")
plt.xlabel("epoch"); plt.ylabel("smooth-L1 loss (embedding space)")
plt.legend(); plt.title("JEPA pretraining loss")
plt.show()